In [1]:
#------------------------------------------------ Begin_Librairie ----------------------------------------
import datetime
import os
import re
import time

import pandas as pd
import requests
from bs4 import BeautifulSoup

from time import sleep

import urllib3
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

In [2]:
#------------------------------------------------ Begin_ fileName ----------------------------------------
regulatorName = 'AI AFSC'

print(f"Running {regulatorName} Web Scraping Tool v.1.1")
now = datetime.datetime.now()
filename = '{} SQL Ready {}.xlsx'.format(regulatorName, str(now).replace(':', '.')[:-7])

scriptfolder = f"C:\\Users\\wuj1\\OneDrive - Moody's\\Desktop\\Regulator\\{regulatorName}"
os.chdir(scriptfolder)
tempfolder = os.path.join(scriptfolder, 'tempfolder')
if os.path.exists(tempfolder):
    for rem in os.listdir(tempfolder):
        os.remove(os.path.join(tempfolder, rem))
else:
    os.mkdir(tempfolder)

Running AI AFSC Web Scraping Tool v.1.1


In [3]:
#------------------------------------------------ Begin_chromedriver ----------------------------------------
# Chrome driver not needed - pages scraped via requests + BeautifulSoup.
# Kept as placeholder in case Selenium is needed for future list types.
# from selenium import webdriver
# chromeOptions = webdriver.ChromeOptions()
# prefs = {
#     'plugins.always_open_pdf_externally': True,
#     'download.prompt_for_download': False,
#     'download.default_directory': tempfolder,
#     'profile.default_content_setting_values.automatic_downloads': 1,
# }
# chromeOptions.add_experimental_option('prefs', prefs)
# driver = webdriver.Chrome(options=chromeOptions)
# driver.maximize_window()

In [4]:
#------------------------------------------------ Begin_Variable ----------------------------------------

processdate = now.strftime('%Y-%m-%d')

regdict = {
    regulatorName + ' 1': 'https://www.fsc.org.ai/banksre.php',
    regulatorName + ' 2': 'https://www.fsc.org.ai/insurancere.php',
    regulatorName + ' 3': 'https://www.fsc.org.ai/companymanagementre.php',
    regulatorName + ' 4': 'https://www.fsc.org.ai/mutualfundsre.php',
    regulatorName + ' 5': 'https://www.fsc.org.ai/msbre.php',
    regulatorName + ' 6': 'https://www.fsc.org.ai/crediture.php',
}

Typology = {
    regulatorName + ' 1': 'Banking Regulated Entities',
    regulatorName + ' 2': 'Insurance Regulated Entities',
    regulatorName + ' 3': 'Company Management Regulated Entities',
    regulatorName + ' 4': 'Mutual Funds',
    regulatorName + ' 5': 'Money Services Business Regulated Entities',
    regulatorName + ' 6': 'Credit Union Regulated Entities',
}

sqldict = {
    'bvdid': [], 'priority': [], 'ListLabel': [], 'Typology': [], 'EntryType': [],
    'Name': [], 'InternalID_1': [], 'InternalID_1_type': [], 'InternalID_2': [],
    'InternalID_2_type': [], 'InternalID_3': [], 'InternalID_3_type': [], 'CoType': [],
    'License_Type': [], 'Address_1': [], 'Address_2': [], 'City': [], 'Zip': [],
    'Cntry': [], 'Phone': [], 'Fax': [], 'Website': [], 'Email': [],
    'RegulationType': [], 'RegulationTypeCode': [], 'RegulationDate': [],
    'CancellationDate': [], 'RegCtry': [], 'RegCode': [], 'ListCode': [],
    'ListLanguage': [], 'ListValidityDate': [], 'ListName': [], 'ListProcessDate': [],
    'LEI Code': [], 'BIC SWIFT Code': [], 'Name - Mother Company': [],
    'Address_1 - Mother company': [], 'Address_2 -  Mother company': [],
    'City - Mother company': [], 'Zip - Mother company': [],
    'Cntry - Mother company': [], 'Phone - Mother company': [], 'Check': [],
}

In [5]:
#------------------------------------------------ Begin_Function ----------------------------------------
HEADERS = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) '
                  'AppleWebKit/537.36 (KHTML, like Gecko) Chrome/143.0.0.0 Safari/537.36'
}


def bourange_same_length_array(sqldict):
    maxlen = len(sqldict['ListProcessDate'])
    for key in sqldict:
        if len(sqldict[key]) != maxlen:
            sqldict[key] = sqldict[key] + [''] * (maxlen - len(sqldict[key]))
    return sqldict


def parse_table(table):
    """Return (headers, rows) from a <table>. Row of only <th> = header; <td> rows = data."""
    headers, rows = [], []
    for tr in table.find_all('tr'):
        ths = tr.find_all('th')
        tds = tr.find_all('td')
        if ths and not tds:
            headers = [th.get_text(' ', strip=True) for th in ths]
        elif tds:
            rows.append([td.get_text(' ', strip=True) for td in tds])
    return headers, rows

In [ ]:
#------------------------------------------------ Begin_Main ----------------------------------------

for k, reg in enumerate(regdict):
    url = regdict[reg]
    print(f'[INFO] : Working {k+1}/{len(regdict)} _({reg})_')

    resp = requests.get(url, headers=HEADERS, timeout=60, verify=False)
    soup = BeautifulSoup(resp.text, 'html.parser')

    list_code = reg.split()[-1]
    list_name = Typology[reg]
    entity_count = 0

    for table in soup.select('div.table-responsive table'):
        section_h = table.find_previous(['h3', 'h4'])
        section = section_h.get_text(' ', strip=True) if section_h else list_name

        headers, rows = parse_table(table)
        headers_low = [h.lower() for h in headers]

        def col(row, *keys):
            for key in keys:
                for i, h in enumerate(headers_low):
                    if key in h and i < len(row):
                        return row[i]
            return ''

        for row in rows:
            name = col(row, 'entity', 'name', 'company', 'licensee')
            if not name:
                continue

            country_raw = col(row, 'country')
            country = 'Anguilla' if 'anguilla' in country_raw.lower() else (country_raw or 'Anguilla')
            city = re.sub(r',?\s*B\.?W\.?I\.?\s*$', '', country_raw, flags=re.I).replace(country, '').strip(' ,')

            entity_count += 1
            sqldict['Name'].append(name)
            sqldict['Address_1'].append(col(row, 'address', 'street') if len(col(row, 'address', 'street'))>2 else '')
            sqldict['City'].append(city)
            sqldict['Cntry'].append(country)
            sqldict['Phone'].append(col(row, 'tel', 'phone') if len(col(row, 'tel', 'phone'))>2 else '')
            sqldict['Fax'].append(col(row, 'fax') if len(col(row, 'fax'))>2 else '')
            sqldict['Email'].append(col(row, 'email'))
            sqldict['Website'].append(col(row, 'website', 'web'))
            # sqldict['License_Type'].append(col(row, 'class', 'licen', 'type'))
            sqldict['Typology'].append(section)
            sqldict['RegCtry'].append('AI')
            sqldict['RegCode'].append('AFSC')
            sqldict['ListCode'].append(list_code)
            sqldict['RegulationType'].append('Regulated')
            sqldict['ListName'].append(list_name)
            sqldict['ListProcessDate'].append(processdate)
            sqldict = bourange_same_length_array(sqldict)

    print(f"  Extracted {entity_count} entities")
    for rem in os.listdir(tempfolder):
        os.remove(os.path.join(tempfolder, rem))
    sleep(1)

print(f"[INFO] Total: {len(sqldict['Name'])}")

[INFO] : Working 1/6 _(AI AFSC 1)_
  Extracted 3 entities
[INFO] : Working 2/6 _(AI AFSC 2)_
  Extracted 96 entities
[INFO] : Working 3/6 _(AI AFSC 3)_
  Extracted 48 entities
[INFO] : Working 4/6 _(AI AFSC 4)_
  Extracted 11 entities
[INFO] : Working 5/6 _(AI AFSC 5)_
  Extracted 4 entities
[INFO] : Working 6/6 _(AI AFSC 6)_
  Extracted 2 entities
[INFO] Total: 164


In [7]:
#------------------------------------------------ Save DataFrame to Excel ----------------------------------------
os.chdir(scriptfolder)
df = pd.DataFrame(sqldict)
df.to_excel(filename, sheet_name='SQL Ready', index=False)
sleep(2)
print(f"[INFO] Excel file '{filename}' saved - {len(df)} rows")

[INFO] Excel file 'AI AFSC SQL Ready 2026-04-21 16.00.47.xlsx' saved - 164 rows
